In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from sft import *
from datetime import datetime, timedelta
from interpolation import interp1d
from scipy.ndimage import gaussian_filter

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [2]:
s = np.load('fluxes_car.npz')

flux_density = s['flux_density']
lati = s['lati'].astype(float)
dates = np.array([datetime.fromisoformat(date) for date in s['date']])

dlat = lati[1] - lati[0]
lat = (lati[1:] + lati[:-1]) / 2

In [3]:
plt.figure(figsize=(10,8))
plt.plot(lat, flux_density[25])
plt.ylim(-10,10)
plt.tight_layout()

In [33]:
plt.figure(figsize=(10,8))
plt.imshow(flux_density.T, 'bwr', aspect='auto', origin='lower', vmin=-10, vmax=10)#, interpolation='bicubic')
plt.tight_layout()

In [42]:
R = 695e8
u = 800
d = 100e10
lam0 = 90
lam1 = 35
lam2 = 70


xi = lati * np.pi / 180 * R
vi = u * np.where(np.abs(lati) < lam0, np.sin(np.pi * lati / lam0), 0)
li = 2 * np.pi * R * np.cos(lati  * np.pi / 180)

y = flux_density[18].copy()
#y[np.abs(lat) > lam2] = -0.1

Q = []
for t in np.arange(dates[18], dates[23], timedelta(days=1)):
    dt = timedelta(days=1).total_seconds()

    y = advect(y, xi, vi, dt, li)
    y = diffuse(y, xi, d, dt, li)

    y_ = interp1d(flux_density, dates, t)

    y[np.abs(lat) < lam1] = y_[np.abs(lat) < lam1]

    Q.append(y)

Q = np.array(Q)


In [43]:
dates_ = np.arange(dates[18], dates[23], timedelta(days=1))
flux_density_ = interp1d(flux_density, dates, dates_)


plt.figure(figsize=(10,8))
#plt.plot(lat, flux_density_[0])
plt.plot(lat, flux_density_[-1])
plt.plot(lat, Q[-1])
plt.xlim(-90,90)
plt.ylim(-15,15)
plt.grid(True)
plt.tight_layout()

In [19]:
plt.figure(figsize=(10,8))
plt.imshow(Q.T, 'bwr', aspect='auto', origin='lower', vmin=-10, vmax=10)
plt.tight_layout()

In [21]:
dates[18]

datetime.datetime(2025, 5, 9, 15, 20, 3)